# Day 18: Recommendation Logic through Association

In [ ]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

data = {
'TransactionID': [1, 1, 1, 2, 2, 3, 3, 3, 4, 4, 5, 5],
'Activity': ['Coding', 'Chess', 'Pizza', 'Coding', 'Chess',
             'Hiking', 'Photography', 'Pizza', 'Hiking', 'Photography', 'Coding',
             'Pizza']
}

df = pd.DataFrame(data)

basket = (df.groupby(['TransactionID', 'Activity'])['Activity']
          .count().unstack().reset_index().fillna(0)
          .set_index('TransactionID'))

def encode_units(x):
    return 1 if x >= 1 else 0

basket_sets = basket.map(encode_units)
print(basket_sets.head())

In [ ]:
frequent_itemsets = apriori(basket_sets, min_support=0.2, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)

rules = rules.sort_values('confidence', ascending=False)

print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())

In [ ]:
premium_rules = rules[(rules['lift'] > 1.5) & (rules['confidence'] > 0.6)]
top3 = premium_rules.sort_values('lift', ascending=False).head(3)
print(top3[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

## Reflection

If we find a strong association between 'Yoga' and 'Juice Tasting,' but the Support is only 0.01 (1%), this rule is **not reliable** for a broad marketing campaign.

Support of 1% means only 1 in 100 transactions contain both activities together. Even if the Confidence and Lift are high, the rule applies to an extremely small fraction of users. Running a marketing campaign based on such a rare pattern risks wasted spend and irrelevant recommendations for 99% of the audience. A reliable rule for mass campaigns typically requires Support ≥ 5–10% to ensure the pattern is statistically meaningful and affects enough users to justify the investment. For niche or hyper-personalized segments this low-support rule could still be useful, but it should never anchor a broad promotional strategy.